# Faruq-v3 ACMC1-HCR — Hard-Competitor Ranking Seed-42 Screening

Eksperimen terakhir setelah residual-error audit. Arsitektur inferensi tetap ACMC1; satu-satunya perubahan adalah training-only hard-competitor pairwise softplus loss pada one-to-one classification branch. Tidak ada pasangan confusion validation yang di-hard-code. Test tetap terkunci.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/acmc1-hard-competitor-ranking'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone = ['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(1,4):
    result = subprocess.run(clone)
    if result.returncode == 0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt == 3: raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO}[dev]'], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
print('REPO  :', REPO)
print('BRANCH:', BRANCH)


In [ ]:
tests = ['tests/test_acmc1_hard_competitor_ranking.py','tests/test_ambiguity_multilevel.py']
subprocess.run([sys.executable,'-m','pytest','-q',*tests], cwd=REPO, check=True)
print('LOCAL COLAB PRE-FLIGHT: PASS')


In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

assert torch.cuda.is_available(), 'Aktifkan T4 GPU atau GPU Colab lain.'
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
    'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/acmc1_optimization_control_seed42.json',
))
ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
D0_CHECKPOINT = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt')
ACMC1_CONTROL = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/acmc1_optimization_control_seed42.json')
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
GROUPED_SUMMARY = DATA_ROOT / 'faruq_grouped_summary.json'
OUTPUT_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-acmc1h-hard-competitor-screening-v1'
if not GROUPED_SUMMARY.is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
assert GROUPED_SUMMARY.is_file(), GROUPED_SUMMARY
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh tersedia.'
last = OUTPUT_ROOT / 'ACMC1H_seed42/weights/last.pt'
best = OUTPUT_ROOT / 'ACMC1H_seed42/weights/best.pt'
print('GPU    :', torch.cuda.get_device_name(0))
print('ACMC1H :', 'COMPLETE' if best.is_file() else ('RESUME' if last.is_file() else 'START'))
print('CONTROL:', ACMC1_CONTROL)
print('OUTPUT :', OUTPUT_ROOT)


In [ ]:
command = [
    sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_acmc1h_screening',
    '--data-root',str(DATA_ROOT),
    '--grouped-summary',str(GROUPED_SUMMARY),
    '--d0-checkpoint',str(D0_CHECKPOINT),
    '--acmc1-control-summary',str(ACMC1_CONTROL),
    '--output-root',str(OUTPUT_ROOT),
    '--seed','42','--device','0','--authorize-training',
]
print('MENJALANKAN:', ' '.join(command), flush=True)
process = subprocess.Popen(command, cwd=REPO, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f'ACMC1-HCR screening gagal, return code={return_code}; traceback lengkap tercetak di atas.')


In [ ]:
import json, pandas as pd
from IPython.display import display

SUMMARY = OUTPUT_ROOT / 'val_reports/acmc1h_seed42_screening.json'
assert SUMMARY.is_file(), f'Screening belum selesai: {SUMMARY}'
result = json.loads(SUMMARY.read_text(encoding='utf-8'))
assert result['evaluation_split'] == 'val'
assert result['test_images_accessed'] is False
assert result['test_opened'] is False
metrics = ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')
rows = [{'model': key, **value} for key,value in result['results'].items()]
display(pd.DataFrame(rows).style.format({name:'{:.2%}' for name in metrics}))
print('ACMC1H vs D0FT :', result['deltas_acmc1h_vs_d0ft'])
print('ACMC1H vs ACMC1:', result['deltas_acmc1h_vs_acmc1'])
print('CRITERIA:', result['criteria'])
print('DECISION:', result['decision'])
print('NEXT    :', result['next_action'])
print('SUMMARY :', SUMMARY)
if result['decision'] == 'FAIL':
    print('STOP: keep ACMC1. Jangan jalankan seed123/2026 dan jangan buka test.')
else:
    print('PASS screening: paired 3-seed boleh dibuat sebagai langkah berikutnya; test tetap terkunci.')
